# Prediction Model

## Imports

In [3]:
import json
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense, Concatenate, Flatten
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import f1_score, precision_score, recall_score

## Load Training Data

In [4]:
data_file = "./data/processed/training_data.json"

with open(data_file, "r", encoding="utf-8") as f:
    training_data = json.load(f)

print(f"Loaded {len(training_data)} training examples")


Loaded 1400 training examples


## Prepare Tokenizers

We create tokenizers for IT skills, soft skills, and designations.

In [5]:
it_skill_texts = []
soft_skill_texts = []
designation_texts = []

for example in training_data:
    it_skill_texts.append(" ".join(example["it_skill_categories"]))
    soft_skill_texts.append(" ".join(example["soft_skills"]))
    designation_texts.append(example["desired_designation"])

# Tokenizers
it_tokenizer = Tokenizer(oov_token="<OOV>")
it_tokenizer.fit_on_texts(it_skill_texts)
NUM_IT_SKILLS = len(it_tokenizer.word_index) + 1

soft_tokenizer = Tokenizer(oov_token="<OOV>")
soft_tokenizer.fit_on_texts(soft_skill_texts)
NUM_SOFT_SKILLS = len(soft_tokenizer.word_index) + 1

designation_tokenizer = Tokenizer(oov_token="<OOV>")
designation_tokenizer.fit_on_texts(designation_texts)
NUM_DESIGNATIONS = len(designation_tokenizer.word_index) + 1

print(f"IT skills: {NUM_IT_SKILLS}, Soft skills: {NUM_SOFT_SKILLS}, Designations: {NUM_DESIGNATIONS}")


IT skills: 142, Soft skills: 111, Designations: 31


## Prepare Input Sequences

In [6]:
MAX_IT_LEN = max(len(s.split()) for s in it_skill_texts)
MAX_SOFT_LEN = max(len(s.split()) for s in soft_skill_texts)

# Convert to sequences
it_sequences = it_tokenizer.texts_to_sequences(it_skill_texts)
soft_sequences = soft_tokenizer.texts_to_sequences(soft_skill_texts)
designation_sequences = designation_tokenizer.texts_to_sequences(designation_texts)

# Pad sequences
X_it = pad_sequences(it_sequences, maxlen=MAX_IT_LEN, padding='post')
X_soft = pad_sequences(soft_sequences, maxlen=MAX_SOFT_LEN, padding='post')
X_designation = np.array([seq[0]-1 if len(seq)>0 else 0 for seq in designation_sequences])  # integer indices

print(f"Shapes: IT: {X_it.shape}, Soft: {X_soft.shape}, Designation: {X_designation.shape}")


Shapes: IT: (1400, 154), Soft: (1400, 21), Designation: (1400,)


## Prepare Output Labels

We convert next_skill and next_soft_skill into weighted multi-hot vectors.

In [7]:
NUM_EXAMPLES = len(training_data)

Y_it = np.zeros((NUM_EXAMPLES, NUM_IT_SKILLS), dtype=np.float32)
Y_soft = np.zeros((NUM_EXAMPLES, NUM_SOFT_SKILLS), dtype=np.float32)

for i, example in enumerate(training_data):
    # next IT skills
    for skill, count in example["next_skill"].items():
        idx = it_tokenizer.word_index.get(skill)
        if idx:
            Y_it[i, idx] = count  # use count as weight

    # next soft skills
    for skill, count in example["next_soft_skill"].items():
        idx = soft_tokenizer.word_index.get(skill)
        if idx:
            Y_soft[i, idx] = count

print(f"Output shapes: IT: {Y_it.shape}, Soft: {Y_soft.shape}")


Output shapes: IT: (1400, 142), Soft: (1400, 111)


## Train-Test Split

In [8]:
X_it_train, X_it_test, X_soft_train, X_soft_test, X_des_train, X_des_test, Y_it_train, Y_it_test, Y_soft_train, Y_soft_test = train_test_split(
    X_it, X_soft, X_designation, Y_it, Y_soft, test_size=0.2, random_state=42
)

print(f"Train samples: {X_it_train.shape[0]}, Test samples: {X_it_test.shape[0]}")


Train samples: 1120, Test samples: 280


## Build LSTM Model

In [11]:
EMB_DIM = 64
LSTM_UNITS = 128

# Inputs
it_input = Input(shape=(MAX_IT_LEN,), name="it_input")
soft_input = Input(shape=(MAX_SOFT_LEN,), name="soft_input")
des_input = Input(shape=(1,), name="designation_input")

# Embeddings
it_emb = Embedding(NUM_IT_SKILLS, EMB_DIM, mask_zero=True)(it_input)
soft_emb = Embedding(NUM_SOFT_SKILLS, EMB_DIM, mask_zero=True)(soft_input)
des_emb = Embedding(NUM_DESIGNATIONS, EMB_DIM)(des_input)
des_flat = Flatten()(des_emb)

# Concatenate embeddings
x = Concatenate()([Flatten()(it_emb), Flatten()(soft_emb), des_flat])
x = Dense(256, activation="relu")(x)

# Output layers
it_output = Dense(NUM_IT_SKILLS, activation="sigmoid", name="next_skill")(x)
soft_output = Dense(NUM_SOFT_SKILLS, activation="sigmoid", name="next_soft_skill")(x)

# Model
model = Model(inputs=[it_input, soft_input, des_input], outputs=[it_output, soft_output])
model.compile(
    optimizer=Adam(0.001),
    loss={"next_skill": "binary_crossentropy", "next_soft_skill": "binary_crossentropy"},
    metrics={"next_skill": "accuracy", "next_soft_skill": "accuracy"}
)

model.summary()


/Users/hammadhassan/.local/share/virtualenvs/resume_processor-Ef8iKj0V/lib/python3.13/site-packages/keras/src/layers/layer.py:982: UserWarning: Layer 'flatten_4' (of type Flatten) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(
/Users/hammadhassan/.local/share/virtualenvs/resume_processor-Ef8iKj0V/lib/python3.13/site-packages/keras/src/layers/layer.py:982: UserWarning: Layer 'flatten_5' (of type Flatten) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ it_input            │ (None, 154)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ soft_input          │ (None, 21)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ designation_input   │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_3         │ (None, 154, 64)   │      9,088 │ it_input[0][0]    │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_4         │ (None, 21, 64)    │      7,104 │ soft_input[0][0]  │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_5         │ (None, 1, 64)     │      1,984 │ designation_inpu… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_4 (Flatten) │ (None, 9856)      │          0 │ embedding_3[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_5 (Flatten) │ (None, 1344)      │          0 │ embedding_4[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_3 (Flatten) │ (None, 64)        │          0 │ embedding_5[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_1       │ (None, 11264)     │          0 │ flatten_4[0][0],  │
│ (Concatenate)       │                   │            │ flatten_5[0][0],  │
│                     │                   │            │ flatten_3[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 256)       │  2,883,840 │ concatenate_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ next_skill (Dense)  │ (None, 142)       │     36,494 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ next_soft_skill     │ (None, 111)       │     28,527 │ dense_1[0][0]     │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 2,967,037 (11.32 MB)

 Trainable params: 2,967,037 (11.32 MB)

 Non-trainable params: 0 (0.00 B)

## Train the Model

In [12]:
history = model.fit(
    [X_it_train, X_soft_train, X_des_train],
    [Y_it_train, Y_soft_train],
    validation_data=([X_it_test, X_soft_test, X_des_test], [Y_it_test, Y_soft_test]),
    epochs=10,
    batch_size=64
)


Epoch 1/10
18/18 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - loss: 0.4213 - next_skill_accuracy: 0.0000e+00 - next_skill_loss: 0.1915 - next_soft_skill_accuracy: 0.2170 - next_soft_skill_loss: 0.2198 - val_loss: 0.0629 - val_next_skill_accuracy: 0.0000e+00 - val_next_skill_loss: 1.8357e-04 - val_next_soft_skill_accuracy: 0.3357 - val_next_soft_skill_loss: 0.0634
Epoch 2/10
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0608 - next_skill_accuracy: 0.0000e+00 - next_skill_loss: 1.7129e-04 - next_soft_skill_accuracy: 0.3554 - next_soft_skill_loss: 0.0606 - val_loss: 0.0542 - val_next_skill_accuracy: 0.0000e+00 - val_next_skill_loss: 1.6122e-05 - val_next_soft_skill_accuracy: 0.3357 - val_next_soft_skill_loss: 0.0544
Epoch 3/10
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0537 - next_skill_accuracy: 0.0000e+00 - next_skill_loss: 4.3887e-05 - next_soft_skill_accuracy: 0.3714 - next_soft_skill_loss: 0.0534 - val_loss: 0.0596 - val_next_skill_accuracy: 0.0000e+00 - val_next_skill_loss: 9.2294

## Evaluate the Model

In [15]:
# Cell 9: Evaluate using F1, Precision, Recall

# Predict on test set
Y_it_pred, Y_soft_pred = model.predict([X_it_test, X_soft_test, X_des_test])

# Binarize predictions
Y_it_pred_bin = (Y_it_pred > 0.5).astype(int)
Y_soft_pred_bin = (Y_soft_pred > 0.5).astype(int)

print(Y_it_pred)

# Ensure test labels are integers too
Y_it_test_bin = (Y_it_test > 0).astype(int)
Y_soft_test_bin = (Y_soft_test > 0).astype(int)

# Compute metrics
f1_it = f1_score(Y_it_test_bin, Y_it_pred_bin, average="micro")
precision_it = precision_score(Y_it_test_bin, Y_it_pred_bin, average="micro")
recall_it = recall_score(Y_it_test_bin, Y_it_pred_bin, average="micro")

f1_soft = f1_score(Y_soft_test_bin, Y_soft_pred_bin, average="micro")
precision_soft = precision_score(Y_soft_test_bin, Y_soft_pred_bin, average="micro")
recall_soft = recall_score(Y_soft_test_bin, Y_soft_pred_bin, average="micro")

print("IT Skill Metrics:")
print(f"F1: {f1_it:.3f}, Precision: {precision_it:.3f}, Recall: {recall_it:.3f}")

print("Soft Skill Metrics:")
print(f"F1: {f1_soft:.3f}, Precision: {precision_soft:.3f}, Recall: {recall_soft:.3f}")


9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
[[1.6101305e-13 7.7281577e-11 2.6573809e-11 ... 1.3555422e-18
  4.5108808e-07 2.7912752e-06]
 [2.4182613e-15 4.1635272e-12 1.3464605e-12 ... 7.4090206e-20
  2.5560848e-07 2.1175083e-06]
 [1.2286469e-11 1.9597205e-06 1.7350651e-10 ... 1.8447835e-15
  1.8267930e-06 2.7256127e-07]
 ...
 [1.4816218e-11 1.3688909e-08 1.0514115e-10 ... 7.7385466e-17
  1.3108039e-06 1.6473934e-06]
 [1.2557983e-08 3.4254106e-06 1.3251443e-07 ... 1.9825073e-12
  6.8068762e-06 4.9503824e-06]
 [4.7103807e-14 2.4296516e-09 1.2342606e-11 ... 1.8793017e-17
  3.1445495e-07 1.9688191e-07]]
IT Skill Metrics:
F1: 0.000, Precision: 0.000, Recall: 0.000
Soft Skill Metrics:
F1: 0.712, Precision: 0.728, Recall: 0.697


/Users/hammadhassan/.local/share/virtualenvs/resume_processor-Ef8iKj0V/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/hammadhassan/.local/share/virtualenvs/resume_processor-Ef8iKj0V/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/hammadhassan/.local/share/virtualenvs/resume_processor-Ef8iKj0V/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 due to no true samples. Use `zero_divis

## Save the model

In [18]:
model.save("../model/lstm_model.h5")

import json
with open("../model/it_tokenizer.json", "w") as f:
    json.dump(it_tokenizer.to_json(), f)
with open("../model/soft_tokenizer.json", "w") as f:
    json.dump(soft_tokenizer.to_json(), f)
with open("../model/designation_tokenizer.json", "w") as f:
    json.dump(designation_tokenizer.to_json(), f)
